# **LASSO regression**

Marek Šugár

In this notebook we aim to optimize regularization parameter $\lambda$ for every single stock ticker included in database. Using this we can optimize the overall performance more however, needs to be strongly checked on testing as we are balancing on the edge of overfit, of course.

In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from DataFramePrep import generate_TrainingDataFrame

# Metrics to measure successfulness
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.preprocessing import StandardScaler

# ML Stuff
from sklearn.linear_model import Lasso

# In case of convergence problem, supress warning
import warnings

warnings.filterwarnings("ignore")

In [2]:
TrainingDataFrame, tickers, historic_columns, StockDataDatabase = generate_TrainingDataFrame()

# **LASSO regression**

In [3]:
performance_tracker_regularization = {}

for ticker in tickers["Ticker"]:
    performance_tracker_regularization[ticker] = []
    for Alpha in np.arange(4.0, 4.9, 0.01):
        Stock_Data = pd.read_sql(
            f"SELECT * FROM StockData WHERE Ticker='{ticker}' AND Date>='2017-09-14' AND Date<='2022-12-31'",
            con=StockDataDatabase, parse_dates=["Date"]
        )
            
        Stock_Data = pd.merge(Stock_Data, TrainingDataFrame, on="Date")
        Stock_Data["Target"] = Stock_Data["Close"]
        Stock_Data = Stock_Data.dropna().reset_index(drop=True)

        training_length = 4
        prediction_length = 1

        MAPEs = []

        for window_start in range(len(Stock_Data) - (training_length + prediction_length)):
            # Define feature and target windows
            Training_Features = Stock_Data[historic_columns].iloc[window_start:window_start+training_length]
            Training_Target = Stock_Data["Target"].iloc[window_start:window_start+training_length]

            Test_Features = Stock_Data[historic_columns].iloc[
                window_start+training_length : window_start+training_length+prediction_length
            ]
            Test_Target = Stock_Data["Target"].iloc[
                window_start+training_length : window_start+training_length+prediction_length
            ]

            # Skip empty slices
            if Training_Features.empty or Test_Features.empty:
                continue
            
            # Scale only selected features
            scaler = StandardScaler()
            Training_Features = scaler.fit_transform(Training_Features)
            Test_Features = scaler.transform(Test_Features)

            # Fit KNN
            MODEL = Lasso(alpha=Alpha)
            MODEL.fit(Training_Features, Training_Target)

            # Predict and evaluate
            prediction = MODEL.predict(Test_Features)
            MAPEs.append(100 * mean_absolute_percentage_error(Test_Target, prediction))
            
        # Save results
        performance_tracker_regularization[ticker].append((Alpha, np.mean(MAPEs)))
        print(Alpha, np.mean(MAPEs))


4.0 1.5628550825383374
4.01 1.5627607182448084
4.02 1.562654879297606
4.029999999999999 1.5625409308053129
4.039999999999999 1.5624135963504362
4.049999999999999 1.5622741170598589
4.059999999999999 1.5621257954926053
4.0699999999999985 1.5619792877016312
4.079999999999998 1.561859226135384
4.089999999999998 1.561742695047014
4.099999999999998 1.561650892810889
4.109999999999998 1.561576792535949
4.119999999999997 1.5615020598808442
4.129999999999997 1.561424097481853
4.139999999999997 1.5613691771225218
4.149999999999997 1.5613345781789947
4.159999999999997 1.5612927899820952
4.169999999999996 1.5613981002706143
4.179999999999996 1.5615132423861022
4.189999999999996 1.561627320834475
4.199999999999996 1.561742270964417
4.2099999999999955 1.561847324545488
4.219999999999995 1.5619440621471383
4.229999999999995 1.562040095806056
4.239999999999995 1.5621340438877624
4.249999999999995 1.5622221332162995
4.2599999999999945 1.562322403748845
4.269999999999994 1.562492204904578
4.27999999999

In [5]:
parameters = performance_tracker_regularization.copy()
optimal = {}

for ticker in tickers["Ticker"]:
    best_ones = sorted(parameters[ticker], key=lambda x: x[1])
    print(ticker, best_ones[0][0])

ACN 4.159999999999997
ADBE 4.899999999999981
AMD 4.0
AKAM 4.0
APH 4.0
ADI 4.0
AAPL 4.0
AMAT 4.0
ANET 4.0
ADSK 4.0
AVGO 4.0
CDNS 4.0
CDW 4.0
CSCO 4.0
CTSH 4.0
GLW 4.0
DELL 4.0
ENPH 4.049999999999999
EPAM 4.899999999999981
FFIV 4.0
FICO 4.899999999999981
FSLR 4.049999999999999
FTNT 4.0
IT 4.0
GEN 4.0
GDDY 4.219999999999995
HPE 4.0
HPQ 4.0
IBM 4.0
INTC 4.119999999999997
INTU 4.899999999999981
JBL 4.0
KEYS 4.0
KLAC 4.899999999999981
LRCX 4.0
MCHP 4.0
MU 4.0
MSFT 4.0
MPWR 4.899999999999981
MSI 4.0
NTAP 4.0
NVDA 4.0
NXPI 4.0
ON 4.0
ORCL 4.339999999999993
PANW 4.0
PTC 4.0
QCOM 4.0
ROP 4.5399999999999885
CRM 4.0
STX 4.0
NOW 4.899999999999981
SWKS 4.0
SMCI 4.0
SNPS 4.0
TEL 4.0
TDY 4.0
TER 4.0
TXN 4.179999999999996
TRMB 4.0
TYL 4.899999999999981
VRSN 4.619999999999987
WDC 4.0
WDAY 4.0
ZBRA 4.899999999999981
